[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:
import torch
import math

In [20]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    B, S, D = Q.shape
    output = torch.zeros(B, S, D)
    for i in range(0, S, block_size):
      qi = Q[:, i:i+block_size]
      bs_q = qi.shape[1]
      row_max = torch.full((B, bs_q, 1), -1e9, device=Q.device)
      row_sum = torch.zeros(B, bs_q, 1, device=Q.device)
      acc = torch.zeros(B, bs_q, D, device=Q.device)
      for j in range(0, S, block_size):
        kj = K[:,j:j+block_size]
        vj = V[:,j:j+block_size]
        score = qi @ kj.transpose(-2, -1)/torch.sqrt(torch.tensor(D))
        block_max = score.max(dim=-1, keepdim=True).values
        new_max = torch.maximum(block_max, row_max)
        exp_score = torch.exp(score-new_max)
        correction = torch.exp(row_max-new_max)
        row_sum = row_sum * correction + exp_score.sum(dim=-1,keepdim=True)
        acc = acc * correction + exp_score @ vj
        row_max = new_max
      output[:, i:i+bs_q] = acc/row_sum
    return output

In [21]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [22]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (22.7ms)
  ✅ [2/4] Non-aligned block size (2.7ms)
  ✅ [3/4] Block size invariant (3.0ms)
  ✅ [4/4] Gradient flow (37.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (65.6ms total)
  Progress saved. Run status() to see your dashboard.

